
# Stage A — Build Reusable Grid → Polygon Mapping (Simple, Cell-Center)

This notebook creates a **reusable mapping** from ERA5 grid points `(latitude, longitude)` to **basin** and **watershed** polygons using a **cell-center point-in-polygon** approach.

**Why a separate mapping?** Once generated, it can be reused for ERA5, ECMWF/GraphCast forecasts, or any other dataset on the same grid.

**Outputs** (written to `data/spatial/grid_mapping/` by default):
- `era5_grid_to_polygons.parquet`
- `era5_grid_to_polygons.csv`
- `era5_grid_to_polygons_QA.txt` (quick summary)



## 1. Environment & Requirements

> Run this cell once to ensure dependencies are available (uncomment if needed).


In [1]:

# !pip install pandas geopandas shapely pyproj fiona huggingface_hub

In [2]:

# Robust project root resolver so notebook works no matter where it's run from
from pathlib import Path
import os, subprocess

def get_project_root(max_up=6):
    # Try git first
    try:
        root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    # Fallback: climb until we find a directory containing /data and /code
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "data").exists() and (p / "code").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root resolved to:", PROJECT_ROOT)


Project root resolved to: /Users/liuq13/bhutan_climate_modeling



## 2. Configuration

Adjust paths only if your repository structure differs.


In [3]:
import os
import glob
import textwrap
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

# --- INPUTS ---
# Merged ERA5 grid-level parquet (already produced by your merge notebook; timestamps aren't used here)
ERA5_MERGED_PARQUET = str(PROJECT_ROOT / "data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet")

# Polygon directories (basins and watersheds) inside your repository
BASINS_DIR      = str(PROJECT_ROOT / "data/boundaries/basins")
WATERSHEDS_DIR  = str(PROJECT_ROOT / "data/boundaries/186_watershed")

# --- OUTPUTS ---
OUT_DIR = str(PROJECT_ROOT / "data" / "spatial" / "grid_mapping")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
OUT_BASENAME = "era5_grid_to_polygons"  # -> {OUT_BASENAME}.parquet/.csv and a QA .txt

In [4]:
# Candidate ID columns to auto-detect polygon IDs (fallback creates sequential IDs)
CANDIDATE_ID_COLS = [
    "basin_id","BASIN_ID","Basin_ID","ws_id","WS_ID","watershed_id","Watershed_ID",
    "OBJECTID","FID","ID","id","NAME","Name","name","CODE","code"
]

# Optional: fall back to Hugging Face if the local parquet is missing
USE_HF_IF_LOCAL_MISSING = True
HF_REPO_ID   = "qlk0610/bhutan-climate"  # your public dataset repo
HF_REPO_FILE = "era5/era5_merged/merged_era5_6hour_1979_2025.parquet"
# At runtime we'll set ERA5_SOURCE_PATH = ERA5_MERGED_PARQUET unless we need to fetch from HF
ERA5_SOURCE_PATH = ERA5_MERGED_PARQUET

In [5]:

# Resolve source path: use local parquet if present; otherwise, optionally fetch from Hugging Face
import os
if (not os.path.exists(ERA5_MERGED_PARQUET)) and USE_HF_IF_LOCAL_MISSING:
    try:
        from huggingface_hub import hf_hub_download
        print("Local parquet not found. Fetching from Hugging Face…")
        ERA5_SOURCE_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_REPO_FILE)
        print("Downloaded to:", ERA5_SOURCE_PATH)
    except Exception as e:
        raise FileNotFoundError(f"Local parquet missing and HF download failed: {e}")
else:
    ERA5_SOURCE_PATH = ERA5_MERGED_PARQUET
print("Using ERA5 parquet:", ERA5_SOURCE_PATH)


Using ERA5 parquet: /Users/liuq13/bhutan_climate_modeling/data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet



## 3. Helper Functions


In [6]:

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def pick_vector_file(dir_path: str):
    """Pick the first vector file (.shp/.gpkg) in a directory (prefers basin/watershed-like names)."""
    candidates = []
    candidates += glob.glob(os.path.join(dir_path, "*.shp"))
    candidates += glob.glob(os.path.join(dir_path, "*.gpkg"))
    if not candidates:
        raise FileNotFoundError(f"No .shp or .gpkg found in {dir_path}")
    preferred = sorted(
        candidates,
        key=lambda p: (0 if any(k in os.path.basename(p).lower() for k in ["basin","watershed","wsh","catch","subbasin"]) else 1, p)
    )
    return preferred[0]

def read_polygons(dir_path: str) -> gpd.GeoDataFrame:
    vec = pick_vector_file(dir_path)
    gdf = gpd.read_file(vec)
    if gdf.crs is None:
        # If CRS is missing, assume WGS84; change if your data differ
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    # choose an ID column or create one
    id_col = None
    for c in CANDIDATE_ID_COLS:
        if c in gdf.columns:
            id_col = c
            break
    if id_col is None:
        id_col = "poly_id"
        gdf[id_col] = range(1, len(gdf)+1)
    return gdf[[id_col, "geometry"]].rename(columns={id_col: "polygon_id"})

def stable_grid_id(df_latlon: pd.DataFrame) -> pd.Series:
    """Stable, collision-resistant ID from (lat, lon)."""
    hashed = pd.util.hash_pandas_object(df_latlon[["latitude","longitude"]], index=False)
    return "g" + hashed.astype("uint64").map(lambda x: format(x, "016x"))

def make_points_gdf(latlon: pd.DataFrame) -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(
        latlon.copy(),
        geometry=[Point(lon, lat) for lat, lon in zip(latlon["latitude"], latlon["longitude"])],
        crs=4326
    )

def spatial_join_id(points: gpd.GeoDataFrame, polys: gpd.GeoDataFrame, label: str) -> pd.Series:
    """Return Series of polygon IDs for `label` using 'within', then fallback to 'intersects' if needed."""
    joined = gpd.sjoin(points, polys, how="left", predicate="within")
    s = joined["polygon_id"].rename(f"{label}_id")
    if s.isna().mean() > 0.1:  # fallback if many misses
        joined2 = gpd.sjoin(points, polys, how="left", predicate="intersects")
        s2 = joined2["polygon_id"].rename(f"{label}_id")
        s = s.fillna(s2)
    return s



## 4. Load Unique Grid Points from Merged ERA5 Parquet


In [7]:

# Read only latitude/longitude columns to build the unique grid
if not os.path.exists(ERA5_SOURCE_PATH):
    raise FileNotFoundError(f"Parquet not found: {ERA5_SOURCE_PATH}")

df_latlon = pd.read_parquet(ERA5_SOURCE_PATH, columns=["latitude","longitude"]).dropna(subset=["latitude","longitude"])
grid = df_latlon.drop_duplicates(subset=["latitude","longitude"]).reset_index(drop=True)
grid["grid_id"] = stable_grid_id(grid)
grid = grid[["grid_id","latitude","longitude"]]
print(f"Unique grid cells: {len(grid):,}")
grid.head()


Unique grid cells: 135


,grid_id,latitude,longitude
0,gf81d7fb303616195,26.5,88.50
1,g0be12287883e2881,26.5,88.75
2,gcaddc3e21bd4a5f1,26.5,89.00
3,g6aa345cebf65e0f4,26.5,89.25
4,g55221829641bde3e,26.5,89.50



## 5. Load Basin & Watershed Polygons (Standardize CRS to EPSG:4326)


In [8]:
print(f"Loading basins from: {BASINS_DIR}")
basins = read_polygons(BASINS_DIR)
print(f"Basins loaded: {len(basins):,}")
basins.head()

Loading basins from: /Users/liuq13/bhutan_climate_modeling/data/boundaries/basins
Basins loaded: 10


,polygon_id,geometry
0,1,"POLYGON ((90.35281 27.22062, 90.35316 27.22078..."
1,2,"POLYGON ((92.03864 27.25695, 92.03829 27.25595..."
2,3,"POLYGON ((90.77251 27.94007, 90.7727 27.93836,..."
3,4,"POLYGON ((88.94369 26.93431, 88.94641 26.93632..."
4,5,"POLYGON ((89.30275 26.84244, 89.30086 26.84241..."


In [9]:
print(f"Loading watersheds from: {WATERSHEDS_DIR}")
watersheds = read_polygons(WATERSHEDS_DIR)
print(f"Watersheds loaded: {len(watersheds):,}")
watersheds.head()

Loading watersheds from: /Users/liuq13/bhutan_climate_modeling/data/boundaries/186_watershed
Watersheds loaded: 186


,polygon_id,geometry
0,1,"POLYGON ((92.04047 27.25524, 92.03816 27.25527..."
1,2,"POLYGON ((89.74885 28.1869, 89.74993 28.18672,..."
2,3,"POLYGON ((89.62562 28.16421, 89.6273 28.16388,..."
3,4,"POLYGON ((89.80726 28.23634, 89.80923 28.23521..."
4,5,"POLYGON ((90.14051 28.19408, 90.14257 28.19301..."



## 6. Point-in-Polygon Assignment (Cell-Center)
This step assigns each grid cell to a **basin** and a **watershed**.  
If many cells remain unassigned (NaN), we fallback from `within` to `intersects` to account for slivers/precision issues.


In [10]:
points_gdf = make_points_gdf(grid)

print("Assigning basin IDs…")
basin_ids = spatial_join_id(points_gdf, basins, label="basin")

print("Assigning watershed IDs…")
watershed_ids = spatial_join_id(points_gdf, watersheds, label="watershed")

mapping = points_gdf.drop(columns="geometry").copy()
mapping["basin_id"] = basin_ids.values
mapping["watershed_id"] = watershed_ids.values
mapping["outside_flag"] = mapping["basin_id"].isna() & mapping["watershed_id"].isna()

mapping.head()


Assigning basin IDs…
Assigning watershed IDs…


,grid_id,latitude,longitude,basin_id,watershed_id,outside_flag
0,gf81d7fb303616195,26.5,88.50,NaN,NaN,True
1,g0be12287883e2881,26.5,88.75,NaN,NaN,True
2,gcaddc3e21bd4a5f1,26.5,89.00,NaN,NaN,True
3,g6aa345cebf65e0f4,26.5,89.25,NaN,NaN,True
4,g55221829641bde3e,26.5,89.50,NaN,NaN,True



## 7. Save Mapping Artifacts


In [11]:

ensure_dir(OUT_DIR)
out_parquet = os.path.join(OUT_DIR, f"{OUT_BASENAME}.parquet")
out_csv     = os.path.join(OUT_DIR, f"{OUT_BASENAME}.csv")
qa_txt      = os.path.join(OUT_DIR, f"{OUT_BASENAME}_QA.txt")

mapping.to_parquet(out_parquet, index=False)
mapping.to_csv(out_csv, index=False)

outside_n = int(mapping["outside_flag"].sum())
summary = textwrap.dedent(f"""
=== GRID → POLYGON MAPPING (simple, cell-center) ===
Input parquet:  {ERA5_MERGED_PARQUET}
Basins dir:     {BASINS_DIR}
Watersheds dir: {WATERSHEDS_DIR}

Unique grid cells:   {len(mapping):,}
Unmapped (outside):  {outside_n:,}  ({outside_n/len(mapping):.2%})

Example rows:
{mapping.head(5).to_string(index=False)}
""").strip()
print(summary)

with open(qa_txt, "w", encoding="utf-8") as f:
    f.write(summary + "\n")

out_parquet, out_csv, qa_txt


=== GRID → POLYGON MAPPING (simple, cell-center) ===
Input parquet:  /Users/liuq13/bhutan_climate_modeling/data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet
Basins dir:     /Users/liuq13/bhutan_climate_modeling/data/boundaries/basins
Watersheds dir: /Users/liuq13/bhutan_climate_modeling/data/boundaries/186_watershed

Unique grid cells:   135
Unmapped (outside):  61  (45.19%)

Example rows:
          grid_id  latitude  longitude  basin_id  watershed_id  outside_flag
gf81d7fb303616195      26.5      88.50       NaN           NaN          True
g0be12287883e2881      26.5      88.75       NaN           NaN          True
gcaddc3e21bd4a5f1      26.5      89.00       NaN           NaN          True
g6aa345cebf65e0f4      26.5      89.25       NaN           NaN          True
g55221829641bde3e      26.5      89.50       NaN           NaN          True


('/Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/era5_grid_to_polygons.parquet',
 '/Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/era5_grid_to_polygons.csv',
 '/Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/era5_grid_to_polygons_QA.txt')


## 8. Quick QA Preview


In [12]:

mapping.describe(include='all')


,grid_id,latitude,longitude,basin_id,watershed_id,outside_flag
count,135,135.000000,135.000000,74.000000,56.000000,135
unique,135,NaN,NaN,NaN,NaN,2
top,gf81d7fb303616195,NaN,NaN,NaN,NaN,False
freq,1,NaN,NaN,NaN,NaN,74
mean,NaN,27.500000,90.250000,6.310811,89.732143,NaN
std,NaN,0.647901,1.084146,2.006330,51.959247,NaN
min,NaN,26.500000,88.500000,1.000000,6.000000,NaN
25%,NaN,27.000000,89.250000,5.250000,44.750000,NaN
50%,NaN,27.500000,90.250000,7.000000,95.000000,NaN
75%,NaN,28.000000,91.250000,7.750000,136.250000,NaN



## 9. Next Steps

- **Stage B (Aggregation)**: Join this mapping to your merged ERA5 parquet and compute polygon × time (6‑hourly) aggregates for basins and watersheds.
- **Weighted upgrade (later)**: Replace this simple mapping with a fractional overlap weights table and reuse the same Stage B logic.
